# Classification Experiments
ResNet-18 trained on 5 pipeline scenarios with accuracy/precision/recall/F1 evaluation.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import json
import matplotlib.pyplot as plt
import pandas as pd
import torch
from pathlib import Path

from classification.config import ClassificationConfig, TRAINING_SCENARIOS
from classification.dataset import XRayDataModule
from classification.model import get_model
from classification.trainer import Trainer
from evaluation.classification_metrics import ClassificationMetricsEvaluator

logging.basicConfig(level=logging.INFO)
PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'output' / '05_classification'
LABELS_CSV = PROJECT_ROOT / 'dataset' / 'original' / 'balanced_prompts_fixed.csv'

In [ ]:
# Configuration
cfg = ClassificationConfig(
    model_name='resnet18',
    batch_size=32,
    num_epochs=20,
    learning_rate=1e-4,
    patience=5,
    seed=42,
)
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')
print(f'Scenarios: {TRAINING_SCENARIOS}')

In [ ]:
def run_scenario(scenario: str, cfg: ClassificationConfig):
    """Train and evaluate one scenario. Returns (trainer, val_loader, class_names)."""
    cfg.scenario = scenario
    cfg.__post_init__()
    
    image_dir = cfg.resolve_image_dir()
    if not image_dir.exists():
        print(f'  Skipping {scenario}: {image_dir} not found')
        return None
    
    dm = XRayDataModule(
        image_dir=image_dir,
        labels_csv=LABELS_CSV,
        image_size=cfg.image_size,
        batch_size=cfg.batch_size,
        val_split=cfg.val_split,
        seed=cfg.seed,
    )
    train_loader, val_loader = dm.get_loaders()
    cfg.num_classes = dm.num_classes
    
    model = get_model(cfg.model_name, dm.num_classes, cfg.pretrained)
    trainer = Trainer(model, cfg)
    
    out_dir = OUTPUT_DIR / scenario
    history = trainer.train(train_loader, val_loader, out_dir / 'checkpoints')
    
    return trainer, val_loader, dm.class_names, history

In [ ]:
# Run all scenarios
all_results = {}

for scenario in TRAINING_SCENARIOS:
    print(f'\n=== Scenario: {scenario} ===')
    result = run_scenario(scenario, cfg)
    if result is None:
        continue
    trainer, val_loader, class_names, history = result
    
    # Evaluate
    eval_result = trainer.evaluate(val_loader)
    all_results[scenario] = {
        'history': history,
        'eval': eval_result,
        'class_names': class_names,
    }
    print(f'  Accuracy: {eval_result["accuracy"]:.4f}')

In [ ]:
# Plot training curves for each scenario
for scenario, result in all_results.items():
    history = result['history']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'], label='Val')
    axes[0].set_title(f'{scenario} — Loss')
    axes[0].legend()
    
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'], label='Val')
    axes[1].set_title(f'{scenario} — Accuracy')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / scenario / 'training_curves.png', dpi=150)
    plt.show()

In [ ]:
# Confusion matrices
for scenario, result in all_results.items():
    eval_res = result['eval']
    cn = result['class_names']
    evaluator = ClassificationMetricsEvaluator(cn)
    cm_path = OUTPUT_DIR / scenario / 'confusion_matrix.png'
    evaluator.plot_confusion_matrix(
        eval_res['y_true'], eval_res['y_pred'],
        save_path=cm_path, title=f'CM — {scenario}'
    )
    print(f'{scenario}: CM saved')

In [ ]:
# Summary table
from evaluation.classification_metrics import (
    compute_accuracy, compute_precision, compute_recall, compute_f1
)

rows = []
for scenario, result in all_results.items():
    ev = result['eval']
    rows.append({
        'scenario': scenario,
        'accuracy': compute_accuracy(ev['y_true'], ev['y_pred']),
        'precision': compute_precision(ev['y_true'], ev['y_pred']),
        'recall': compute_recall(ev['y_true'], ev['y_pred']),
        'f1': compute_f1(ev['y_true'], ev['y_pred']),
    })

summary_df = pd.DataFrame(rows)
summary_df.to_csv(OUTPUT_DIR / 'scenario_summary.csv', index=False)
print(summary_df.to_string(index=False))